Nexar Collision Prediction — Preprocessing
Extracts frames at 10fps, runs YOLOv26-seg (vehicles) and DepthAnything V2 Large,
saves results as .npy files to Google Drive.

In [1]:
# Cell 1 — Install dependencies
!pip install -q ultralytics transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 28.9 MB/s eta 0:00:00


In [2]:
# Cell 2 — Mount Google Drive and set up paths
from google.colab import drive
drive.mount('/content/drive')

import os

ROOT = '/content/drive/MyDrive/nexar'

PATHS = {
    'train_videos':  os.path.join(ROOT, 'videos', 'train'),
    'test_videos':   os.path.join(ROOT, 'videos', 'test'),
    'frames_train':  os.path.join(ROOT, 'frames', 'train'),
    'frames_test':   os.path.join(ROOT, 'frames', 'test'),
    'seg_train':     os.path.join(ROOT, 'segmentation', 'train'),
    'seg_test':      os.path.join(ROOT, 'segmentation', 'test'),
    'depth_train':   os.path.join(ROOT, 'depth', 'train'),
    'depth_test':    os.path.join(ROOT, 'depth', 'test'),
}

# Create all output folders if they don't exist
for path in PATHS.values():
    os.makedirs(path, exist_ok=True)

print("Drive mounted. Folder structure:")
for key, path in PATHS.items():
    print(f"  {key:20s} -> {path}")

Mounted at /content/drive
Drive mounted. Folder structure:
  train_videos         -> /content/drive/MyDrive/nexar/videos/train
  test_videos          -> /content/drive/MyDrive/nexar/videos/test
  frames_train         -> /content/drive/MyDrive/nexar/frames/train
  frames_test          -> /content/drive/MyDrive/nexar/frames/test
  seg_train            -> /content/drive/MyDrive/nexar/segmentation/train
  seg_test             -> /content/drive/MyDrive/nexar/segmentation/test
  depth_train          -> /content/drive/MyDrive/nexar/depth/train
  depth_test           -> /content/drive/MyDrive/nexar/depth/test


In [4]:
# Cell 3 — Load models
import torch
from ultralytics import YOLO
from transformers import pipeline

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# YOLOv26-seg nano — vehicle segmentation
yolo = YOLO("yolo26n-seg.pt")
yolo.to(DEVICE)
print("YOLO loaded")

# DepthAnything V2 Large — depth estimation
depth_pipe = pipeline(
    task="depth-estimation",
    model="depth-anything/Depth-Anything-V2-Large-hf",
    device=0 if DEVICE == "cuda" else -1,
)
print("DepthAnything V2 Large loaded")

Using device: cpu
YOLO loaded


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/503 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/775 [00:00<?, ?B/s]

The image processor of type `DPTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


DepthAnything V2 Large loaded


In [5]:
# Cell 4 — Helper functions
import cv2
import numpy as np
from PIL import Image

TARGET_FPS = 10
TARGET_W, TARGET_H = 640, 360

# YOLO class IDs for vehicles (COCO): car, motorcycle, bus, truck
VEHICLE_CLASS_IDS = {2, 3, 5, 7}


def extract_frames(video_path):
    """Extract frames at TARGET_FPS, resize to TARGET_W x TARGET_H.
    Returns np.ndarray of shape [N, H, W, 3], dtype uint8."""
    cap = cv2.VideoCapture(video_path)
    src_fps = cap.get(cv2.CAP_PROP_FPS)
    interval = max(1, round(src_fps / TARGET_FPS))

    frames = []
    idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if idx % interval == 0:
            frame = cv2.resize(frame, (TARGET_W, TARGET_H))
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame)
        idx += 1
    cap.release()
    return np.stack(frames).astype(np.uint8)  # [N, H, W, 3]


def extract_seg_masks(frames):
    """Run YOLOv26-seg on each frame, return binary vehicle masks.
    Returns np.ndarray of shape [N, H, W], dtype uint8."""
    masks = []
    for frame in frames:
        result = yolo(frame, verbose=False)[0]
        mask = np.zeros((TARGET_H, TARGET_W), dtype=np.uint8)
        if result.masks is not None:
            for seg_mask, cls_id in zip(result.masks.data, result.boxes.cls):
                if int(cls_id) in VEHICLE_CLASS_IDS:
                    m = seg_mask.cpu().numpy()
                    m = cv2.resize(m, (TARGET_W, TARGET_H), interpolation=cv2.INTER_NEAREST)
                    mask = np.maximum(mask, (m > 0.5).astype(np.uint8))
        masks.append(mask)
    return np.stack(masks)  # [N, H, W]


def extract_depth_maps(frames):
    """Run DepthAnything V2 on each frame, return normalised depth maps.
    Returns np.ndarray of shape [N, H, W], dtype float16."""
    depth_maps = []
    for frame in frames:
        pil_img = Image.fromarray(frame)
        result = depth_pipe(pil_img)
        depth = np.array(result['depth'], dtype=np.float32)
        depth = cv2.resize(depth, (TARGET_W, TARGET_H))
        # Normalise to [0, 1]
        d_min, d_max = depth.min(), depth.max()
        if d_max > d_min:
            depth = (depth - d_min) / (d_max - d_min)
        depth_maps.append(depth.astype(np.float16))
    return np.stack(depth_maps)  # [N, H, W]


print("Helper functions defined.")

Helper functions defined.


In [9]:
# Cell 5 — Sanity check on a single video
import matplotlib.pyplot as plt

# Pick the first train video
test_video_id = sorted(os.listdir(PATHS['train_videos']))[0].replace('.mp4', '')
test_video_path = os.path.join(PATHS['train_videos'], f'{test_video_id}.mp4')
print(f"Testing on: {test_video_path}")

# Extract
frames = extract_frames(test_video_path)
print(f"Frames:       {frames.shape}  dtype={frames.dtype}")

masks = extract_seg_masks(frames)
print(f"Seg masks:    {masks.shape}  dtype={masks.dtype}")

depths = extract_depth_maps(frames)
print(f"Depth maps:   {depths.shape}  dtype={depths.dtype}")

# Visualise frame 0
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].imshow(frames[0]);       axes[0].set_title('RGB frame')
axes[1].imshow(masks[0], cmap='gray'); axes[1].set_title('Seg mask')
axes[2].imshow(depths[0], cmap='plasma'); axes[2].set_title('Depth map')
for ax in axes: ax.axis('off')
plt.tight_layout()
plt.show()

IndexError: list index out of range

If the above output indicates the directory is empty, please upload your training video files (e.g., `.mp4` files) to the specified path in your Google Drive: `/content/drive/MyDrive/nexar/videos/train`. Once uploaded, re-run the `Cell 5` to confirm the fix.